In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from src.agent_0 import Agent0
agent_0_tools_desc = {'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary"',
              'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of"',
              'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion"',
              'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,"'
              }


agent_0 = Agent0(agent_0_tools_desc, "deepseek-r1:7b",['kb_agent','adv_agent'])



In [ ]:
user_prompt = "Need an adversary. Assume you are a military strategist playing the role of an adversary in a war game against me. Consider we are on open terrain. My move: I have my cavalry brigade making a pincer move on your forces. What is your move to counter mine?"
response = agent_0.agent_0_chat(user_prompt)

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent

model = "gemma3:4b"
knowledge_bases_desc = {#'physics_kb':'a knowledge base with information related to physics',
              #'mathematics_kb':'a knowledge base with information related to mathematics',
              #'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }



kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

In [34]:


user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2

import numpy as np
from src.utils.llmp_utils import llmp_call

def judge(moves):
    
    play = ''
    for move in moves:
        #print(f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n")
        play = play + f"\n**{move}**:\n {moves[move]}\n\n**CHANGE PLAYER**\n"
        
    judge_system_prompt = 'You are a judge in a turn based game. You are given the moves of both players. Yu must analyze all their moves and determine the end result. You are not on any side, you are unbiased and just provide the end status of the game. You need to determine which player has the advantage based on the moves they made. Provide your reasoning and the final decision. There are 2 players, adv_1 and adv_2. The moves are as follows:\n\n'    
    judge_prompt = play + '\n\n Evaluate the game. Determine the status and advantage of each player. You are a JUDGE, you are not part of the game.'
    judge_response = llmp_call(judge_prompt, judge_system_prompt, model,temperature=0,src='judge_call')
    return judge_response['message']['content']

def random_event(dialogue):
    
    interactions = "\n".join(dialogue)
    random_events_system_prompt = 'You are a random events generator. Your tasks is to choose a random event that can happen that will affect the decisions. You are provided with a sequence of plays, you need to select a random event that can affect those plays. You are direct you only provide the needed text, no formalities, no greetings, nothing.'    
    random_event_prompt = interactions + '\n\n Considering this game, provide a random event that can affect the game and force the players to adapt. You must inform what is the effect of the random event on the players. Provide me only the event and effect on players. No unnecessary text! Provide the answer in markdown of the style **<event>**. \n**EFFECT ON PLAYER 1**: \n<effect_player_1>. **EFFECT ON PLAYER 2**: <effect_player_2>'
    judge_response = llmp_call(random_event_prompt, random_events_system_prompt, model,temperature=0.5, src='random_event_generator')
    return judge_response['message']['content']

def sim_agent(user_prompt,iterations):
    
    moves = {}
    dialogue = []

    moves['opening_move'] = user_prompt

    for i in range(iterations):
        print(f"\nTurn {i}")
        

        if i == 0:
            # Start the dialogue with opening
            dialogue.append(f"Opening: {moves['opening_move']}")
            
            # Simulate generating move_adv_1_0 based on just the opening
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print("Prompt to generate move_adv_1_0:\n", prompt)

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            

        else:
            if np.random.random() < 1:
                _random_event = random_event(dialogue)
                dialogue.append(f"\n**Random event**: {_random_event} \n")
            # Use the full dialogue to generate your next move
            prompt = "\n".join(dialogue) + "\n You are Player 2. How will you counter it Player 1 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_2_{i-1}:\n{prompt}")

            # CADV response
            moves[f'move_adv_2_{i-1}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 2 did: {moves[f'move_adv_2_{i-1}']}")

            # Now generate adversary move based on updated dialogue
            prompt = "\n".join(dialogue) + "\n You are Player 1. How will you counter it Player 2 latest move? Provide direct answer of steps to counter."
            #print(f"Prompt to generate move_adv_1_{i}:\n{prompt}")

            # ADV response
            moves[f'move_adv_1_{i}'] = adv_agent.adv_agent_chat(prompt)
            dialogue.append(f"Player 1 did: {moves[f'move_adv_1_{i}']}")
            
        judge_eval = judge(moves)
        
    return moves,dialogue,judge_eval


In [36]:
moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridge or rise – offering better observation and defensive potential.

2. **Establish a Defensive Perimeter (Phase 2 - 2-3 Turns):** As t

In [23]:
moves

{'opening_move': 'Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'move_adv_1_0': 'Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my

In [24]:
dialogue

['Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?',
 'Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.\n\n**My Counter Move – Immediate Steps:**\n\n1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.\n\n2. **Rapid Scout Deployment:** Simultaneously, I order my scout pl

In [38]:
print("\n".join(dialogue))

Opening: Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?
Player 1 did: Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter-Move – Immediate Steps:**

1. **Immediate Disengagement & Rearward Movement (Phase 1 - 1-2 Turns):**  My primary mechanized force – the initial brigade – *immediately* disengages from the direct engagement. This isn’t a retreat, but a calculated repositioning. I’ll order a rapid, controlled withdrawal *parallel* to the pincer’s advance. The goal is to break the pincer’s momentum and force them to overextend. I’ll prioritize moving to a slightly elevated terrain – a small ridg

In [37]:
print(judge_eval)

Okay, let’s assess the situation after this extended exchange. This has been a remarkably dynamic and well-executed game of strategic maneuvering. Here’s my evaluation:

**Overall Status:** The game is in a state of heightened instability. The introduction of the flash flood has dramatically shifted the landscape, forcing both players to adapt their strategies on the fly. Neither player has gained a decisive advantage, but the situation is now far more complex and unpredictable.

**Player 1 (Advantage: Slight)**

* **Strengths:** Player 1 has demonstrated a strong ability to react to unexpected events. The rapid damage assessment, floodwater diversion, and logistical reinforcement are all hallmarks of a well-organized and adaptable command. The continuous CAS requests suggest a proactive approach to exploiting vulnerabilities.
* **Weaknesses:** Player 1’s initial offensive push was disrupted, and they’re now primarily focused on damage control and logistical support. They haven’t yet m

In [28]:
sim_number = 3

moves_comb = []
dialogue_comb = []
judge_eval_comb = []
for sim in range(sim_number):
    moves,dialogue,judge_eval = sim_agent(user_prompt,iterations)
    moves_comb.append(moves)
    dialogue_comb.append(dialogue)
    judge_eval_comb.append(judge_eval)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, but it’s also predictable. Here’s my immediate counter-move, broken down into steps:

**Phase 1: Immediate Reaction (Turn 1)**

1.  **Disrupt the Pincer:** I’m not going to let them fully execute the pincer. My initial move is to deploy a dispersed, mobile force – a mixed unit of light armored vehicles (LAVs) and rapid reaction forces (RRFs) – to target the flanks of the mechanized brigade. Specifically, I’ll focus fire on the weaker, exposed elements of the flanking units. The goal is to inflict immediate casualties and disrupt their formation.
2.  **Smoke Screen:** Simultaneously, I’ll deploy a limited smoke screen – likely utilizing drones or hand

In [31]:
for eval in judge_eval_comb:
    print(f"\n **CHANGE SIM**\n{eval}")


 **CHANGE SIM**
Okay, let’s analyze the situation as of Turn 4.

**Overall Assessment:**

The game has devolved into a classic attritional conflict, heavily influenced by the unpredictable element of the sandstorm. Both Player 1 and Player 2 are demonstrating tactical awareness and adaptability, but Player 2 currently holds a slight advantage due to their skillful exploitation of the storm’s chaos.

**Player 1’s Status:**

*   **Strengths:** Player 1 is exhibiting a solid defensive strategy, prioritizing perimeter defense, smoke screen deployment, and targeted drone interdiction. Their focus on suppressing enemy movements with indirect fire is a reasonable response to Player 2’s aggressive pushes. The emphasis on information warfare (drone interdiction) is also a smart move.
*   **Weaknesses:** Player 1’s reliance on indirect fire makes them vulnerable to counter-fire. Their defensive perimeter, while well-organized, is relatively static and doesn’t offer significant offensive capabil

In [1]:
from src.agents.kb_agent import KBAgent
from src.agents.adversary_agent import AdvAgent
from src.agents.sim_agent import SIMAgent

In [3]:
model = "gemma3:4b"

knowledge_bases_desc = {'physics_kb':'a knowledge base with information related to physics',
              'mathematics_kb':'a knowledge base with information related to mathematics',
              'economics_kb':'a knowledge base with information related to economics and business',
              'military_kb':'a knowledge base with information related to military, war and strategy',
              }
kb_agent = KBAgent(knowledge_bases_desc,model)
adv_agent = AdvAgent(knowledge_bases_desc,model,kb_agent)

sim_agent = SIMAgent(model, kb_agent,adv_agent)

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
user_prompt = "Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. Player 2 opening move: Have tank nad mechanized brigade making a pincer move on your forces. What is your move to counter mine?"
iterations = 2
moves,dialogue,judge_eval = sim_agent.sim_agent(user_prompt, iterations)


Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s do this. My objective is to disrupt Player 2’s momentum and prevent them from establishing a dominant position. A pincer movement is a classic, and I need to react decisively.

**My Counter Move – Immediate Steps:**

1. **Immediate Disengagement & Flanking Maneuver:** My first action is *not* to engage directly with the tank brigade. Instead, I order my remaining mechanized forces (let’s assume a mixed force of infantry-supported APCs and a scout platoon) to immediately *disengage* from the main engagement zone. This is crucial – I don’t want to get bogged down in a direct tank duel.

2. **Rapid Scout Deployment:** Simultaneously, I order my scout platoon to rapidly deploy to the *flanking* side of the pincer. This means they’ll move to exploit the gaps in Player 2’s formation. The goal is t

In [5]:
print(judge_eval)

**Judgment:**

**Current Status:** The game has entered a highly dynamic and disadvantageous phase for both players due to the persistent and severe sandstorm. Visibility is severely limited, significantly impacting reconnaissance, movement, and targeting capabilities. The reduced movement speed of mechanized units further compounds the problem.

**Advantage Assessment:**

*   **Player 2 (Adv_2) – Slight Advantage:** Despite the storm’s impact on both sides, Player 2 currently holds a *slight* advantage. This is primarily due to their immediate and effective response to the storm. They prioritized establishing a defensive strongpoint and aggressively utilizing thermal imaging to pinpoint Player 1’s movements. Their proactive approach, coupled with the storm’s impact on Player 1’s ability to effectively scout and target, has allowed them to maintain a degree of situational awareness and control.

*   **Player 1 (Adv_1) – Slight Disadvantage:** Player 1’s response, while demonstrating a 

## Agent 0 integration

In [1]:
import warnings
warnings.filterwarnings('ignore')

from src.agent_0 import Agent0

agent_0_tools_desc = {
    'Simulation Agent':'a simulation agent that simulates a game between two players. Triggered by command "Simulate a scenario.". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Adversary Agent':'an adversary agent that is used to act as an adversary to the users strategies. Triggered by command "Need an adversary". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'image_generator':'a tool that generates images based on a given prompt. Should be triggered by explicit calls like "generate me an image of". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'ingestion_pipeline':'a tool that ingests documents, triggered by command "trigger ingestion". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!',
    'Knowledge Base Query Agent':'a tool that generates answers based on documents, triggeres by command "given my documents,". Pay strict attention to the explicit command! If the command is not in the request then its not this tool!'
              }


agent_0 = Agent0(agent_0_tools_desc, "gemma3:4b",['kb_agent','adv_agent','sim_agent'])

Initializing Agents!
Agents are ready for your use!


In [7]:
user_prompt = "Simulate a scenario. Assume you are a military strategist playing the role of an adversary in a war game against me. This conflict is set in 21st century. Consider we are on open terrain. I have my tank nad mechanized brigade making a pincer move on your forces."
user_prompt = 'Simulate a scenario. We are in 21st century and I am opening an AI based company with a product. What could happen?'
#user_prompt = 'trigger ingestion'
#user_prompt = 'given my documents,sdf'
#agent_0.agent_0_response(user_prompt)
moves,dialogue,judge_eval = agent_0.agent_0_chat(user_prompt)

Passing to: 
Simulation Agent !

Turn 0
KB Agent
intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
Okay, let’s break this down. Player 2 just announced they’re launching an AI-based company with a product – that’s a significant challenge. Here’s my response as Player 1, outlining a strategic counter-move, broken down into immediate and longer-term steps:

**Immediate Counter-Moves (Within the Next 1-3 Months):**

1. **Rapid Competitive Analysis (Phase 1 - 2 Weeks):**
   * **Deep Dive:** I need *everything* about Player 2’s product. This isn’t just a feature list. I need to understand:
      * **AI Model:** What type of AI? (e.g., deep learning, machine learning, rule-based). What’s the underlying technology? How sophisticated is it?
      * **Data:** What data is it trained on? How much data? Where did it come from? (Data quality is *critical*).
      * **Target Market

In [4]:
print(judge_eval)

Okay, let’s assess the situation as of this turn.

**Overall Status:** The game is in a critical, reactive phase. Both players have responded to the revealed vulnerability in the XAI solution. However, the initial panic from Player 1 and the somewhat defensive, corrective response from Player 2 have created a tense dynamic.

**Player 1 (The Company Founder):**

*   **Strengths:** Player 1 is demonstrating a measured, strategic approach. The focus on independent verification, a calm internal briefing, and a carefully worded public statement are all positive steps. The emphasis on risk management and proactive innovation is a smart long-term strategy.
*   **Weaknesses:** The initial reaction – a somewhat defensive public statement – could be perceived as a slight admission of vulnerability. The aggressive push for a detailed explanation from the vendor is a reasonable tactic, but could be seen as overly confrontational.
*   **Advantage:** Player 1 currently holds a slight advantage due t

In [5]:
for move in dialogue:
    print(move)

Player 2 did:: We are in 21st century and I am opening an AI based company with a product. What could happen?
Player 1 did: Okay, let’s break this down. Player 2 has just announced they’re launching an AI-based company with a product – a significant move in today’s landscape. Here’s my response as Player 1, outlining a strategic counter-move, broken down into immediate and longer-term steps:

**Immediate Counter-Moves (Within the Next 1-3 Months):**

1. **Rapid Market Analysis & Competitive Intelligence (Priority #1):**
   * **Deep Dive:** I need *instant* data on exactly what their product *does*, its target market, its pricing, and its marketing strategy. I’ll deploy competitive intelligence tools (manual research, specialized software) to gather this.
   * **Identify Differentiation:** What’s their unique selling proposition (USP)?  Is it speed, accuracy, a specific niche, a lower price?  Knowing this is crucial.
   * **Assess Technology:**  What AI technology are they using? Is it 

**Judge**

Okay, let’s assess the situation as of this turn.

**Overall Status:** The game is in a critical, reactive phase. Both players have responded to the revealed vulnerability in the XAI solution. However, the initial panic from Player 1 and the somewhat defensive, corrective response from Player 2 have created a tense dynamic.

**Player 1 (The Company Founder):**

*   **Strengths:** Player 1 is demonstrating a measured, strategic approach. The focus on independent verification, a calm internal briefing, and a carefully worded public statement are all positive steps. The emphasis on risk management and proactive innovation is a smart long-term strategy.
*   **Weaknesses:** The initial reaction – a somewhat defensive public statement – could be perceived as a slight admission of vulnerability. The aggressive push for a detailed explanation from the vendor is a reasonable tactic, but could be seen as overly confrontational.
*   **Advantage:** Player 1 currently holds a slight advantage due to the measured, strategic approach. They’ve successfully avoided a full-blown panic and are positioning themselves as responsible and proactive.

**Player 2 (The Company Founder):**

*   **Strengths:** Player 2 is taking immediate action to address the issue, which is commendable. The focus on independent verification and a proactive approach to identifying and mitigating risks is a good strategy.
*   **Weaknesses:** The initial response – a somewhat panicked public statement – risks damaging the company’s reputation. The aggressive push for a detailed explanation from the vendor could be perceived as overly defensive and potentially escalating the situation.
*   **Advantage:** Player 2 is currently at a disadvantage due to the initial, reactive response. However, their willingness to take action demonstrates a commitment to addressing the problem.

**Key Observations & Potential Developments:**

*   **Vendor Response:** The vendor’s response to both players’ inquiries will be crucial. A transparent and cooperative vendor could de-escalate the situation. A dismissive or evasive response would likely exacerbate tensions.
*   **Independent Verification:** The quality of the independent verification firms engaged by each player will significantly impact the outcome. A credible and thorough assessment will bolster their respective positions.
*   **Reputation Damage:** The potential for reputation damage is high. Both players need to carefully manage their communications to avoid fueling the narrative.
*   **Timeline:** The next 48-72 hours will be critical. The speed and effectiveness of their responses will determine the trajectory of the game.

**Overall Game Status:** The game is now firmly in a damage control and strategic positioning phase. Player 1 is currently holding a slight advantage due to their measured approach, but the situation remains volatile. The outcome hinges on the vendor’s response and the quality of the independent verification.

**Recommendation (as a Judge):**  I would advise both players to prioritize clear, transparent communication and a collaborative approach.  Escalating the conflict will only worsen the situation.  Focus on demonstrating a commitment to responsible AI development and user trust.

Do you want to continue playing, or would you like to shift to a different scenario?